# QWCT native RAW BSU / mirror review — four real June 2024 cases
This notebook plots **native QuantConnect RAW** D1 prices *only through the prior session* and intraday M5 from actual historical minute observations. It validates that independently retrieved historical levels/crosses match the **already completed** QWCT 2.8 native run. Four algorithmic swing proxies are **NOT manually certified Gerchik levels**. No orders/fills, no forward-shifted selection, no inferred BPU1 from price-range crossings. Run in a **QuantConnect Python Research Notebook**, not as `main.py`. Two cases AON and DELL selected by default.


In [ ]:
# QuantConnect Cloud Research Notebook — not a trading algorithm
# 2024-06 real QC history; displays true native RAW data only in QuantConnect.
from datetime import datetime, timedelta, time
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from QuantConnect.Research import QuantBook
from QuantConnect import Resolution, DataNormalizationMode

# Prior verified June 2.8 original swing-proxy evidence, never a manual BSU label.
CASES = [
    dict(ticker="AON", day="2024-06-12", src="2024-05-10", side="short",
         raw=293.5900, cluster=293.8175, first="09:35", n_cross=2, basis="MATCH"),
    dict(ticker="DELL", day="2024-06-07", src="2024-05-14", side="long",
         raw=128.7500, cluster=128.6750, first="14:15", n_cross=7, basis="MATCH"),
    dict(ticker="CRM", day="2024-06-27", src="2024-06-06", side="short",
         raw=246.7500, cluster=246.0550, first="09:40", n_cross=1, basis="MATCH"),
    dict(ticker="ABT", day="2024-06-07", src="2023-12-04", side="short",
         raw=105.5000, cluster=105.3150, first="10:20", n_cross=1, basis="REBASED"),
]
CASE_CHOICE = ["AON", "DELL"]  # Change to ["AON","DELL","CRM","ABT"] to inspect all.
qb = QuantBook()
qb.set_start_date(2023, 7, 1)
qb.set_end_date(2024, 7, 1)
qb.set_time_zone("America/New_York")
assets = {
    t: qb.add_equity(t, Resolution.MINUTE, fill_forward=False,
                     data_normalization_mode=DataNormalizationMode.RAW).symbol
    for t in [c["ticker"] for c in CASES]
}
print("READ-ONLY RAW equity research. Original evidence: QWCT native run "
      "4b787847b9e5d2405b2fd97a541fa71e.")
print("No market orders, no levels chosen from future D1, no TVX inference.")


In [ ]:
def one_symbol(frame):
    """QuantBook returns a pandas DataFrame, often Symbol x EndTime."""
    if frame is None or frame.empty:
        raise ValueError("Native RAW History returned no candles")
    if isinstance(frame.index, pd.MultiIndex):
        frame = frame.droplevel(0)
    return frame.sort_index().copy()

def et_index(frame):
    idx = pd.DatetimeIndex(pd.to_datetime(frame.index))
    if idx.tz is None:
        idx = idx.tz_localize("America/New_York")
    else:
        idx = idx.tz_convert("America/New_York")
    frame = frame.copy()
    frame.index = idx
    if not frame.index.is_unique:
        raise ValueError("Duplicate Native TradeBar EndTime")
    return frame

def native_raw_windows(case):
    symbol = assets[case["ticker"]]
    target = datetime.strptime(case["day"], "%Y-%m-%d")
    source = datetime.strptime(case["src"], "%Y-%m-%d")
    # D1 end strictly before session; the day being reviewed is never a D1 input.
    d = one_symbol(qb.history(
        symbol, target-timedelta(days=330), target.replace(hour=9,minute=29),
        Resolution.DAILY, fill_forward=False,
        data_normalization_mode=DataNormalizationMode.RAW))
    d = et_index(d)
    if (d.index >= pd.Timestamp(target.replace(hour=9,minute=29),tz="America/New_York")).any():
        raise ValueError("Noncausal D1 history at or after morning decision")
    # A native daily bar often ends 00:00 the NEXT day; map back to its
    # actual session without consulting future prices.
    native_day = [
        ((x-timedelta(microseconds=1)).date() if x.time()==time(0,0)
         else x.date()) for x in d.index]
    d["session_day"] = pd.to_datetime(native_day)
    d = d.set_index("session_day", drop=True)
    d = d.loc[d.index < pd.Timestamp(target.date())].tail(180)
    if len(d)<180:
        raise ValueError("Fewer than 180 original closed daily bars; do not compare to cohort")
    if d.empty or pd.Timestamp(case["src"]) not in d.index:
        raise ValueError("D1 source missing in historical RAW data")
    m1 = one_symbol(qb.history(
        symbol, target.replace(hour=9,minute=30),
        target.replace(hour=16,minute=1),
        Resolution.MINUTE, fill_forward=False, extended_market_hours=False,
        data_normalization_mode=DataNormalizationMode.RAW))
    m1 = et_index(m1)
    m1 = m1.loc[
        (m1.index.date == target.date()) &
        (m1.index.time > time(9,30)) & (m1.index.time <= time(16,0))]
    if not m1.index.is_monotonic_increasing or m1.empty:
        raise ValueError("Empty or unordered native RAW minute history")
    m5 = m1.resample("5min", closed="right", label="right").agg({
        "open":"first", "high":"max", "low":"min",
        "close":"last", "volume":"sum"})
    n_m1 = m1["close"].resample("5min", closed="right", label="right").count()
    m5["native_m1_count"] = n_m1
    m5 = m5.loc[
        (m5.index.date == target.date()) &
        (m5.index.time >= time(9,35)) & (m5.index.time <= time(16,0))]
    m5 = m5.dropna(subset=["open","high","low","close"])
    return d, m1, m5

def review_geometry(case, d, m1, m5):
    src = d.loc[pd.Timestamp(case["src"])]
    field = "high" if case["side"]=="short" else "low"
    actual_source = float(src[field])
    raw_ok = abs(actual_source-case["raw"]) <= 0.0001
    extrema = m5["high"] if case["side"]=="short" else m5["low"]
    # Same half-native-tick equality as research BsuDetector.
    tol = .005000001
    raw = case["raw"]
    exact = (extrema-raw).abs()<=tol
    short = extrema < raw-tol if case["side"]=="short" else extrema > raw+tol
    crossed = ~exact & ~short & (m5["low"]<=raw) & (m5["high"]>=raw)
    beyond = ~exact & ~short & ~crossed
    labels = pd.Series("BEYOND_NO_SPAN",index=m5.index)
    labels.loc[short]="SHORT_OF"
    labels.loc[crossed]="RANGE_CROSSED"
    labels.loc[exact]="EXACT_WICK"
    # For this case, the independently observed approved native detector
    # baseline required a particular number of range-crossing buckets.
    n_cross=int(crossed.sum())
    same_cross=n_cross == case["n_cross"]
    first_cross = m5.loc[crossed].index.min() if n_cross else None
    first_expected = (first_cross is not None and
                      first_cross.strftime("%H:%M")==case["first"])
    expected=pd.date_range(pd.Timestamp(case["day"]+" 09:35",tz="America/New_York"),
                           pd.Timestamp(case["day"]+" 16:00",tz="America/New_York"),
                           freq="5min")
    valid_counts=(m5.index.equals(expected)
                  and int(m5.iloc[-1]["native_m1_count"])==5)
    print("\n==",case["ticker"],case["day"],case["side"],"==")
    print("D1 original:",case["src"],"recorded:",raw,
          "native source:",actual_source,"MATCH" if raw_ok else "MISMATCH")
    print("Source-vs-cluster:",case["cluster"],"ticks:",
          round((case["cluster"]-raw)/.01,2),
          "source basis:",case["basis"])
    print("Real native M1:",len(m1),"derived 5m time buckets:",len(m5),
          "M5 with all five native M1:",int((m5["native_m1_count"]==5).sum()))
    print("Ranges: exact",int(exact.sum()),"short",int(short.sum()),
          "cross",n_cross,"beyond",int(beyond.sum()))
    print("Cross count vs June2.8 native:",same_cross,
          "first crossing:",str(first_cross),
          "first timestamp match:",first_expected,
          "FULL 78 M5:",valid_counts)
    print("MANUAL BSU/TVX status: INDETERMINATE / UNVERIFIED.")
    if not raw_ok or not same_cross or not first_expected or not valid_counts:
        print("WARNING: SOURCE/GEOMETRY CHECK FAILED. Do not mark BSU; "
              "investigate native history timing/normalization.")
    cols=["open","high","low","close","native_m1_count"]
    if first_cross is not None:
        i = m5.index.get_loc(first_cross)
        print("First crossing and following TWO CLOSED M5 candles:")
        display(m5.iloc[max(0,i-1):min(len(m5),i+3)][cols].round(4))
    return labels, bool(raw_ok and same_cross and first_expected and valid_counts)

def show_charts(case, d, m5, labels, checked):
    fig=make_subplots(rows=2,cols=1,
        row_heights=[.44,.56],vertical_spacing=.13,
        subplot_titles=(
          "D1 RAW history AS-OF prior close (never sees the target day)",
          "M5 from current-day native RAW minute data, after close"))
    fig.add_trace(go.Candlestick(
        x=d.index,open=d["open"],high=d["high"],
        low=d["low"],close=d["close"],name="RAW D1"),
        row=1,col=1)
    fig.add_trace(go.Candlestick(
        x=m5.index,open=m5["open"],high=m5["high"],
        low=m5["low"],close=m5["close"],name="RAW M5 from M1"),
        row=2,col=1)
    for row in (1,2):
        fig.add_hline(y=case["raw"],line_color="red",line_width=2,
                      annotation_text="Original RAW pivot",row=row,col=1)
        fig.add_hline(y=case["cluster"],line_color="blue",line_dash="dash",
                      line_width=2,annotation_text="D1 swing median",row=row,col=1)
    if pd.Timestamp(case["src"]) in d.index:
        fig.add_trace(go.Scatter(
            x=[pd.Timestamp(case["src"])],y=[case["raw"]],
            mode="markers",marker=dict(size=12,color="red",symbol="diamond"),
            name="D1 first_idx origin"),row=1,col=1)
    crosses=m5.loc[labels=="RANGE_CROSSED"]
    if not crosses.empty:
        mids=(crosses["high"]+crosses["low"])/2
        fig.add_trace(go.Scatter(
            x=crosses.index,y=mids,mode="markers",
            marker=dict(size=12,color="orange",symbol="x"),
            name="M5 range straddled pivot ≠ BPU1"),row=2,col=1)
    incomplete=m5.loc[m5["native_m1_count"]<5]
    if not incomplete.empty:
        fig.add_trace(go.Scatter(
            x=incomplete.index,y=incomplete["close"],mode="markers",
            marker=dict(size=12,color="black",symbol="circle-open"),
            name="Missing source M1 in 5m bucket"),row=2,col=1)
    fig.update_xaxes(rangeslider_visible=False)
    fig.update_layout(
        height=1000, width=1200,hovermode="x unified",
        title=(case["ticker"]+"  "+case["day"]+" / "
               +("RAW MATCHED" if checked else "CHECK FAILED — DO NOT LABEL")),
        legend=dict(orientation="h",y=1.09))
    fig.show()

for case in CASES:
    if case["ticker"] not in CASE_CHOICE:
        continue
    d,m1,m5=native_raw_windows(case)
    labels,checked=review_geometry(case,d,m1,m5)
    show_charts(case,d,m5,labels,checked)


## Separate read-only mirror audit
Runs a second independent check of each `kind='mirror'` that the **existing** `strategy.levels` returned from frozen D1, verifying source kind, D1 last swing confirmation, actual subsequent separate close-breach and later close-retest, and chronological `valid_from` without any future prices. This is an **audit of a programmatic mirror proxy**, not a certification of Gerchik's chosen level, a BPU1, or a trade.


In [ ]:
from datetime import timedelta
from math import isfinite
"""Independent read-only validation of Qwct D1 mirror candidates as-of session.

Not a Gerchik manual level label, detector, signal gate or trading approval.
Uses existing Level metadata and strictly prior closed D1 OHLC.
"""
from datetime import timedelta
from math import isfinite


def audit_mirror(d1, level, session, swing_k, tolerance, lookback=60):
    """Return a reproducible causal D1 mirror source/breach/retest trace.

    Side must be inferred unambiguously from the ORIGINAL source bar's
    low/high proximity to the D1 cluster price; otherwise fail closed.
    An accepted mirror is still ONLY an automatic cluster-level proxy.
    """
    out = dict(status="UNKNOWN", reason="", original_kind=None,
               source=None, swing_confirm=None, breach=None,
               retest=None, valid_from=None, price=None)
    def finish(status, reason):
        out["status"], out["reason"] = status, reason
        return out
    if (getattr(level, "kind", None) != "mirror" or
            getattr(level, "source", None) != "swing_cluster"):
        return finish("SKIP", "NOT_SWING_MIRROR")
    if (d1 is None or len(d1)<2*swing_k+5 or
            not d1.index.is_unique or not d1.index.is_monotonic_increasing):
        return finish("UNKNOWN", "INVALID_D1_FRAME")
    i, j = level.first_idx, level.last_idx
    if not isinstance(swing_k,int) or swing_k<1 or not(
            swing_k<=i<j and j+swing_k<len(d1)):
        return finish("UNKNOWN", "UNCONFIRMED_SWING_WINDOW")
    price=float(level.price)
    tol=float(tolerance)
    if not all(isfinite(z) and z>0 for z in (price,tol)):
        return finish("UNKNOWN","BAD_PRICE_OR_CLUSTER_TOLERANCE")
    if level.valid_from is None or level.valid_from>session:
        return finish("UNKNOWN","NOT_VALID_BEFORE_SESSION")
    dates=[x.date() for x in d1.index]
    if (dates[-1]>=session or
            any(a>=b for a,b in zip(dates,dates[1:]))):
        return finish("UNKNOWN","D1_NOT_CAUSAL_TO_SESSION")
    source=d1.iloc[i]
    high,low=float(source["high"]),float(source["low"])
    if high<low or not all(isfinite(v) for v in (high,low)):
        return finish("UNKNOWN","INVALID_SOURCE_OHLC")
    is_high=abs(high-price)<=tol
    is_low=abs(low-price)<=tol
    if is_high==is_low:
        return finish("UNKNOWN","ORIGINAL_KIND_AMBIGUOUS")
    kind="resistance" if is_high else "support"
    last_confirm=dates[j+swing_k]
    base_from=last_confirm+timedelta(days=1)
    if (level.formed_at!=dates[j] or
            level.valid_from<=last_confirm):
        return finish("UNKNOWN","LEVEL_CHRONOLOGY_MISMATCH")
    out.update(original_kind=kind,source=str(dates[i]),
               swing_confirm=str(last_confirm),
               price=price,valid_from=str(level.valid_from))
    start=max(j+1,len(d1)-lookback,1)
    while start<len(d1) and dates[start]<base_from:
        start+=1
    breached=None
    for idx in range(start,len(d1)):
        b=d1.iloc[idx]
        if kind=="support":
            confirmed=(breached is not None and float(b["high"])>=price
                       and float(b["close"])<price)
            if not confirmed and float(b["close"])<price and breached is None:
                breached=idx
        else:
            confirmed=(breached is not None and float(b["low"])<=price
                       and float(b["close"])>price)
            if not confirmed and float(b["close"])>price and breached is None:
                breached=idx
        if confirmed:
            out["breach"]=str(dates[breached])
            out["retest"]=str(dates[idx])
            if dates[idx]+timedelta(days=1)!=level.valid_from:
                return finish("UNKNOWN","RETEST_DATE_DISAGREES_WITH_VALID_FROM")
            return finish("PROXY_VERIFIED","CLOSED_D1_BREACH_RETEST_ASOF")
    return finish("UNKNOWN","NO_SEPARATE_CAUSAL_BREACH_AND_RETEST")


def mirror_inventory(d1, levels, session, swing_k, tolerance, lookback=60):
    """Use only the supplied frozen D1 and user-supplied original Levels."""
    return [(level, audit_mirror(d1,level,session,swing_k,tolerance,lookback))
            for level in levels if level.kind=="mirror"]


In [ ]:
# Optional mirror inventory using original repository's as-of level detection
# and an independent D1 breach/retest chronological validator (above).
try:
    from strategy.config import StrategyConfig
    from strategy.levels import levels_for_session, native_cluster_tolerance
    cfg=StrategyConfig()
    for case in CASES:
        if case["ticker"] not in CASE_CHOICE:
            continue
        d, _, _=native_raw_windows(case)
        day=datetime.strptime(case["day"],"%Y-%m-%d").date()
        detected=levels_for_session(d,day,cfg)
        mirrors=[x for x in detected if x.kind=="mirror"]
        mirrors.sort(key=lambda x:abs(float(x.price)-case["raw"]))
        tol=native_cluster_tolerance(d,cfg)
        print("\\nD1 mirror inventory AS-OF",case["ticker"],case["day"],
              "reported by existing levels.py:",len(mirrors))
        if tol is None:
            print("No actual ATR cluster tolerance, no audit.")
            continue
        for lvl in mirrors[:8]:
            a=audit_mirror(d,lvl,day,cfg.swing_k,tol,cfg.mirror_lookback)
            print("mirror_price",round(float(lvl.price),4),
                  "valid_from",lvl.valid_from,
                  "source",a["source"],"original_kind",a["original_kind"],
                  "breach",a["breach"],"retest",a["retest"],
                  "CAUSAL_AUDIT",a["status"],a["reason"])
        if not mirrors:
            print("No as-of mirror in this day's existing qualified D1 levels.")
    print("\\nAny PROXY_VERIFIED mirror is STILL NOT a manually certified Gerchik BSU.")
except (ImportError, AttributeError) as exc:
    print("Can't import existing strategy.levels in this QuantBook project:",exc)
    print("The native RAW D1/M5 charts above remain usable; mirror status UNKNOWN.")
